# Paper 1 — Multi-seed experiments (Chicago benchmark)Runs 8 spatio-temporal models (HA, LSTM, STGCN, GraphWaveNet, DCRNN, MTGNN, InformerOnly, HASI-Net) across 5 seeds plus the component ablation, on the Chicago women-centric crime panel (77 community areas x 120 months x 4 unified crimes). PSO hyperparameter search runs once and is frozen across seeds. All artifacts are written to results/ and are resume-safe — re-run the cells to continue after a Colab disconnect; completed seeds are skipped.**Runtime: GPU (T4 or better).** Expect ~1-2 hours total (PSO + 5 seeds x 8 models + ablation).

## 0. Environment

In [ ]:
# Install dependencies (Colab GPU runtime).
import subprocess, sys
def pip(pkg): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
for p in ["torch", "numpy", "pandas", "requests", "geopandas", "shapely", "matplotlib", "pyarrow"]:
    try: __import__(p)
    except ImportError: pip(p)
print("deps ready" )

In [ ]:
# Mount Drive and put the repo on the path.
# Put your spatio-temporal-crime checkout on Google Drive (e.g. drag the folder
# into MyDrive), or clone it. The package must be importable as hasi_net.
import sys, pathlib, os
from google.colab import drive
drive.mount("/content/drive")

REPO = pathlib.Path("/content/drive/MyDrive/spatio-temporal-crime")
if not (REPO / "src" / "hasi_net").exists():
    REPO = pathlib.Path("/content/spatio-temporal-crime")
if not (REPO / "src" / "hasi_net").exists():
    raise SystemExit("Repo not found. Upload spatio-temporal-crime to MyDrive or clone it into /content.")
sys.path.insert(0, str(REPO / "src"))
os.chdir(str(REPO))
print("Repo:", REPO)

In [ ]:
# Config (Paper 1, Chicago monthly).
from hasi_net.config import Config, MADHYA_PRADESH, RESULTS_DIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
cfg = Config(
    target_region=MADHYA_PRADESH, use_chicago_benchmark=True,
    chicago_year_start=2015, chicago_year_end=2024,
    device="cuda", lookback=12, horizon=3,
    epochs=80, batch_size=64, lr=1e-3,
    hidden_dim=64, n_graph_layers=2, n_attn_heads=4,
    loss_type="log1p_mse", pso_enabled=True)
print("Config:"); print(cfg.to_dict())

## 1. Build the Chicago panel (downloads ~775k incidents on first run, cached thereafter)

In [ ]:
from hasi_net.data import build_chicago_panel
panel = build_chicago_panel(cfg)
print("Counts [T, N, C]:", panel.counts.shape)
print("Categories:", panel.categories)
import numpy as np
for j, c in enumerate(panel.categories):
    x = panel.counts[:, :, j]
    print(f"  {c:24s} total={int(x.sum())} zero%={100*(x==0).mean():.1f}")

## 2. Multi-seed run: 8 models x 5 seeds (PSO once, frozen)

In [ ]:
from hasi_net.multiseed import run_multiseed
SEEDS = [42, 1, 2, 3, 4]
meanstd, perseed = run_multiseed("chicago", cfg, seeds=SEEDS,
                                 tag="p1_chicago", pso=True, verbose=True)
import pandas as pd
meanstd.round(4)

## 3. Component ablation x 5 seeds

In [ ]:
from hasi_net.multiseed import run_ablation_multiseed
abl = run_ablation_multiseed("chicago", cfg, seeds=SEEDS, tag="p1_chicago",
                             epochs=40, verbose=True)
abl.round(4)

## 4. Persist results to DriveAll CSVs/JSONs/PNGs are already under results/ (inside the repo on Drive). Verify they are there; download the folder for the paper-writing stage.

In [ ]:
import pathlib, json
res = pathlib.Path(REPO) / "results"
arts = sorted(p.name for p in res.glob("*p1_chicago*"))
print("P1 artifacts:")
for a in arts: print(" ", a)
print("
Total results dir size:", sum(p.stat().st_size for p in res.iterdir())//1024, "KB")